# Parsing de Documentos — Parte 4: Documentos Complexos

**Problema:** documentos reais têm hierarquia (títulos, subtítulos, seções),
misturam texto com tabelas, e a **estrutura importa** para o pipeline downstream
(chunking semântico, RAG com contexto hierárquico).

PyMuPDF e pdfplumber não distinguem um título de um parágrafo.

**Ferramentas:**
- `Docling` — converte documentos em Markdown hierárquico
- `Unstructured` — particiona documentos em blocos tipados

In [ ]:
!apt-get install -y poppler-utils -q
!pip install PyMuPDF reportlab docling "unstructured[pdf]" pi-heif -q

In [ ]:
import fitz
import io
import re
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, ListFlowable, ListItem
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER

---
### Criando um PDF complexo para teste

Vamos criar um documento com hierarquia real: títulos, subtítulos, parágrafos,
listas e tabela — tudo no mesmo PDF.

In [ ]:
# Criar PDF complexo com hierarquia
buffer = io.BytesIO()
doc = SimpleDocTemplate(buffer, pagesize=A4,
                        topMargin=50, bottomMargin=50,
                        leftMargin=60, rightMargin=60)
styles = getSampleStyleSheet()

# Estilos customizados
titulo_style = ParagraphStyle('TituloPrincipal', parent=styles['Title'],
                               fontSize=18, spaceAfter=20, alignment=TA_CENTER)
h2_style = ParagraphStyle('H2', parent=styles['Heading2'],
                           fontSize=14, spaceBefore=16, spaceAfter=8)
h3_style = ParagraphStyle('H3', parent=styles['Heading3'],
                           fontSize=12, spaceBefore=12, spaceAfter=6)
body_style = ParagraphStyle('Body', parent=styles['Normal'],
                             fontSize=10, spaceAfter=8, leading=14)

conteudo = []

# Título
conteudo.append(Paragraph("RELATÓRIO TRIMESTRAL DE DESEMPENHO", titulo_style))
conteudo.append(Paragraph("Empresa XYZ Ltda — Q1/2026", styles['Normal']))
conteudo.append(Spacer(1, 20))

# Seção 1
conteudo.append(Paragraph("1. Resumo Executivo", h2_style))
conteudo.append(Paragraph(
    "O primeiro trimestre de 2026 apresentou crescimento de 15% na receita líquida "
    "em comparação ao Q4/2025. A margem operacional manteve-se estável em 22%, "
    "refletindo o controle efetivo de custos implementado no segundo semestre de 2025.",
    body_style))
conteudo.append(Paragraph(
    "Os principais destaques incluem a expansão para o mercado nordestino e o "
    "lançamento de 3 novos produtos na linha premium.",
    body_style))

# Seção 2 com subseções
conteudo.append(Paragraph("2. Indicadores Financeiros", h2_style))
conteudo.append(Paragraph("2.1 Receita por Região", h3_style))
conteudo.append(Paragraph(
    "A região Sudeste continua como principal mercado, representando 58% da receita total. "
    "O Nordeste foi destaque com crescimento de 32% após a abertura de 2 filiais.",
    body_style))

# Tabela
dados = [
    ["Região",    "Q4/2025",       "Q1/2026",       "Variação"],
    ["Sudeste",   "R$ 4.200.000",  "R$ 4.830.000",  "+15%"],
    ["Sul",       "R$ 1.800.000",  "R$ 1.980.000",  "+10%"],
    ["Nordeste",  "R$ 950.000",    "R$ 1.254.000",  "+32%"],
    ["Centro-Oeste","R$ 600.000",  "R$ 660.000",    "+10%"],
    ["TOTAL",     "R$ 7.550.000",  "R$ 8.724.000",  "+15.5%"],
]
t = Table(dados, colWidths=[100, 100, 100, 70])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#34495e')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 9),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ALIGN', (1,0), (-1,-1), 'RIGHT'),
    ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor('#ecf0f1')),
    ('FONTNAME', (0,-1), (-1,-1), 'Helvetica-Bold'),
]))
conteudo.append(Spacer(1, 10))
conteudo.append(t)
conteudo.append(Spacer(1, 10))

# Subseção 2.2
conteudo.append(Paragraph("2.2 Margem Operacional", h3_style))
conteudo.append(Paragraph(
    "A margem operacional de 22% reflete a política de contenção de custos: "
    "redução de 8% em despesas administrativas e renegociação de contratos de fornecedores. "
    "O custo de aquisição de clientes (CAC) reduziu de R$ 340 para R$ 285.",
    body_style))

# Seção 3
conteudo.append(Paragraph("3. Recursos Humanos", h2_style))
conteudo.append(Paragraph(
    "O quadro de funcionários cresceu de 142 para 158 colaboradores (+11%). "
    "Principais movimentações:",
    body_style))

items = [
    "12 novas contratações na área de tecnologia",
    "4 contratações na equipe comercial (Nordeste)",
    "Turnover voluntário: 3.2% (meta: < 5%)",
    "Programa de desenvolvimento: 89% de adesão",
]
lista = ListFlowable(
    [ListItem(Paragraph(item, body_style)) for item in items],
    bulletType='bullet', start='•'
)
conteudo.append(lista)

# Seção 4
conteudo.append(Paragraph("4. Próximos Passos", h2_style))
conteudo.append(Paragraph(
    "Para o Q2/2026, as prioridades são: consolidar operação no Nordeste, "
    "lançar plataforma digital de vendas B2B (previsão: junho/2026), e "
    "implementar sistema de CRM integrado (orçamento aprovado: R$ 450.000).",
    body_style))

doc.build(conteudo)
pdf_complexo_bytes = buffer.getvalue()

# Salvar em disco
caminho_complexo = '/content/relatorio_trimestral.pdf'
with open(caminho_complexo, 'wb') as f:
    f.write(pdf_complexo_bytes)
print(f"PDF complexo criado: {len(pdf_complexo_bytes):,} bytes → {caminho_complexo}")

In [ ]:
# Primeiro, ver como PyMuPDF extrai (tudo flat, sem hierarquia)
doc_mu = fitz.open(caminho_complexo)
texto_flat = doc_mu[0].get_text()
doc_mu.close()

print("=" * 60)
print("PyMuPDF — TEXTO FLAT (sem hierarquia)")
print("=" * 60)
print(texto_flat)
print("\n⚠️  Títulos, parágrafos e itens de lista estão todos no mesmo nível.")
print("   Não há como distinguir 'Resumo Executivo' (seção) de um parágrafo comum.")

---
## Bloco 4a — Docling: parsing com hierarquia em Markdown

Docling analisa o layout do documento e gera saída em **Markdown**,
respeitando a hierarquia: `#` para títulos, `##` para subtítulos, tabelas formatadas.

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
resultado = converter.convert(caminho_complexo)
markdown_docling = resultado.document.export_to_markdown()

print("=" * 60)
print("DOCLING — MARKDOWN HIERÁRQUICO")
print("=" * 60)
print(markdown_docling)

In [ ]:
# Comparação: PyMuPDF (flat) vs Docling (hierárquico)
print("=" * 60)
print("COMPARAÇÃO: PyMuPDF vs Docling")
print("=" * 60)

# Contar níveis de hierarquia no Markdown
titulos_md = re.findall(r'^(#{1,4})\s+(.+)$', markdown_docling, re.MULTILINE)
print("\nHierarquia detectada pelo Docling:")
for nivel, titulo in titulos_md:
    indent = "  " * (len(nivel) - 1)
    print(f"  {indent}{nivel} {titulo}")

print(f"\n→ PyMuPDF: texto corrido, 0 níveis de hierarquia")
print(f"→ Docling: {len(titulos_md)} títulos/subtítulos detectados com hierarquia")
print(f"\nPor que isso importa para RAG?")
print(f"  Com hierarquia, o chunking pode manter cada seção como um chunk semântico,")
print(f"  preservando contexto (ex: '2.1 Receita por Região' + sua tabela).")

---
## Bloco 4b — Docling com modelo de layout

Para documentos mais complexos (colunas múltiplas, cabeçalhos/rodapés,
figuras com legendas), Docling pode usar um modelo pequeno de detecção de layout.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Configurar pipeline com detecção de layout via modelo
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True    # modelo para estrutura de tabelas

converter_model = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

resultado_model = converter_model.convert(caminho_complexo)
markdown_model = resultado_model.document.export_to_markdown()

print("=" * 60)
print("DOCLING + MODELO DE LAYOUT")
print("=" * 60)
print(markdown_model)

# Comparar com o nativo
print("\n" + "=" * 60)
print("COMPARAÇÃO: Docling nativo vs Docling + modelo")
print("=" * 60)
print(f"Docling nativo:  {len(markdown_docling)} chars")
print(f"Docling + model: {len(markdown_model)} chars")
print("\nO modelo pode detectar melhor tabelas, separar cabeçalhos de rodapés,")
print("e lidar com layouts de múltiplas colunas — ao custo de mais tempo de processamento.")

---
## Bloco 4c — Unstructured: particionamento inteligente

Unstructured detecta automaticamente o **tipo** de cada bloco do documento:
`Title`, `NarrativeText`, `Table`, `ListItem`, `Image`, etc.

In [ ]:
from unstructured.partition.pdf import partition_pdf

# Processar o mesmo documento
elementos = partition_pdf(caminho_complexo, strategy="auto")

print("=" * 60)
print("UNSTRUCTURED — ELEMENTOS TIPADOS")
print("=" * 60)
for i, elem in enumerate(elementos):
    tipo = type(elem).__name__
    texto = str(elem)[:100]
    print(f"[{i:2d}] {tipo:20s} | {texto}")

print(f"\n→ Total de elementos: {len(elementos)}")

# Contar por tipo
from collections import Counter
tipos = Counter(type(e).__name__ for e in elementos)
print("\nDistribuição por tipo:")
for tipo, qtd in tipos.most_common():
    print(f"  {tipo:20s}: {qtd}")

In [ ]:
# Comparar abordagens: Docling (Markdown) vs Unstructured (elementos)
print("=" * 60)
print("COMPARAÇÃO: Docling vs Unstructured")
print("=" * 60)

print("\n--- Docling: saída em Markdown hierárquico ---")
print(markdown_docling[:500])
print("...")

print("\n--- Unstructured: elementos tipados ---")
for elem in elementos[:8]:
    print(f"  [{type(elem).__name__}] {str(elem)[:80]}")
print("...")

print("\n→ Docling: ideal para gerar Markdown pronto para chunking semântico em RAG")
print("→ Unstructured: ideal quando você precisa filtrar por tipo (só tabelas, só títulos)")

---
## Bloco 4d — Regex no pós-processamento

Mesmo com Docling/Unstructured, regex continua útil para **transformar** o resultado:
extrair dados estruturados, normalizar formatos e dividir em chunks semânticos.

In [ ]:
# Pós-processamento com regex no Markdown do Docling
print("=" * 60)
print("REGEX NO PÓS-PROCESSAMENTO")
print("=" * 60)

# 1. Extrair valores monetários e converter para float
valores_raw = re.findall(r'R\$\s?([\d.,]+)', markdown_docling)
valores_float = []
for v in valores_raw:
    numero = v.replace('.', '').replace(',', '.')  # "4.200.000" → "4200000"
    try:
        valores_float.append(float(numero))
    except ValueError:
        pass

print("\n1) EXTRAÇÃO + NORMALIZAÇÃO DE VALORES MONETÁRIOS")
print(f"   Strings encontradas: {valores_raw}")
print(f"   Convertidos p/ float: {valores_float}")
print(f"   Maior valor: R$ {max(valores_float):,.2f}")
print(f"   Soma total:  R$ {sum(valores_float):,.2f}")

# 2. Extrair percentuais com contexto (o que veio antes do %)
percentuais_contexto = re.findall(r'(\S+(?:\s+\S+){0,3})\s+(\+?-?\d+[.,]?\d*%)', markdown_docling)
print(f"\n2) PERCENTUAIS COM CONTEXTO")
for contexto, pct in percentuais_contexto:
    print(f"   {pct:>8s} ← ...{contexto}")

# 3. Chunking semântico por seção usando regex nos headers do Markdown
print(f"\n3) CHUNKING POR SEÇÃO (split nos headers)")
chunks = re.split(r'(?=^#{1,3}\s)', markdown_docling, flags=re.MULTILINE)
chunks = [c.strip() for c in chunks if c.strip()]

for i, chunk in enumerate(chunks):
    primeira_linha = chunk.split('\n')[0]
    print(f"   Chunk {i}: ({len(chunk):>4d} chars) {primeira_linha[:60]}")

print(f"\n   → {len(chunks)} chunks semânticos prontos para embeddings/RAG")
print(f"   → Cada chunk preserva o título da seção como contexto")

---
## Trade-offs

| Ferramenta | ✅ Vantagens | ❌ Limitações |
|---|---|---|
| **Docling (nativo)** | Markdown hierárquico, rápido, respeita estrutura | Pode errar em layouts muito complexos |
| **Docling + modelo** | Melhor detecção de layout, colunas, figuras | Mais lento, requer modelo adicional |
| **Unstructured** | Auto-detecção de tipos de bloco, flexível | Configuração mais verbosa, resultado varia por estratégia |

> Docling e Unstructured são para sistemas que **precisam de estrutura**.
> Para um script rápido, PyMuPDF basta. Para um RAG em produção
> com chunking semântico, Docling/Unstructured fazem diferença.
>
> **Mas todos assumem que o PDF tem texto digital.**
> E quando o documento é uma **imagem escaneada**?